# Clase 026 — MultiIndex

**Parte 0** · VanderPlas cap. 3 § 3.6.

> 🎯 Índices jerárquicos cuando hay estructura natural. Aplanar cuando complica.

> ⏱️ ~75 min

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)

## 🧠 Intuición previa

Pensá un **MultiIndex** como **carpetas anidadas** en tu explorador de archivos: la carpeta `España/` contiene las subcarpetas `2023/` y `2024/`, y `Chile/` tiene las suyas. Igual que abrís `España > 2024` para llegar a un archivo, con `df.loc[('España', 2024)]` bajás por los niveles del índice hasta la fila exacta. El nivel externo **agrupa**; el interno **detalla**.

## 1️⃣ Motivación

Datos con jerarquía natural (país→ciudad, año→mes, sector→empresa) caben en un índice de 1 nivel **aplanando**, pero el MultiIndex deja explícita la estructura y facilita el slicing por nivel.

In [ ]:
# 3 formas de construir un MultiIndex

# (a) Desde tuplas
idx_a = pd.MultiIndex.from_tuples([
    ('España', 2023), ('España', 2024),
    ('Chile', 2023),  ('Chile', 2024),
])

# (b) Desde arrays
idx_b = pd.MultiIndex.from_arrays([
    ['España', 'España', 'Chile', 'Chile'],
    [2023, 2024, 2023, 2024],
], names=['país', 'año'])

# (c) Producto cartesiano (más limpio si todas las combinaciones existen)
idx_c = pd.MultiIndex.from_product([['España', 'Chile'], [2023, 2024]], names=['país', 'año'])

print('idx_c:')
print(idx_c)

## 2️⃣ DataFrame con MultiIndex

In [ ]:
df = pd.DataFrame({
    'ventas'  : [100, 120, 80, 95],
    'clientes': [50, 65, 40, 48],
}, index=idx_c)
print(df)

## 3️⃣ Indexación jerárquica

In [ ]:
# Por primer nivel
print('df.loc["España"]:')
print(df.loc['España'])

# Por tupla completa
print('\ndf.loc[("España", 2024)]:')
print(df.loc[('España', 2024)])

# Slice por segundo nivel con xs
print('\ndf.xs(2024, level="año"):')
print(df.xs(2024, level='año'))

## 4️⃣ `unstack` y `stack` — pivot rápido

- `unstack(level)` mueve un nivel del **index** a **columnas** (wide format).
- `stack(level)` lo opuesto (long format).

Útil para visualización rápida.

In [ ]:
wide = df.unstack(level='año')   # años como columnas
print('wide (unstack):')
print(wide)

long = wide.stack(future_stack=True)   # vuelve a long
print('\nde vuelta a long (stack):')
print(long)

## 5️⃣ groupby produce MultiIndex automáticamente

Cuando agrupas por 2+ columnas, pandas devuelve un MultiIndex:

In [ ]:
# Demo sin penguins (sintético)
ventas = pd.DataFrame({
    'tienda' : ['A', 'A', 'B', 'B', 'A', 'B'],
    'mes'    : ['ene', 'feb', 'ene', 'feb', 'ene', 'feb'],
    'monto'  : [100, 120, 80, 95, 110, 90],
})

agg = ventas.groupby(['tienda', 'mes'])['monto'].sum()
print('groupby (MultiIndex Series):')
print(agg)
print(f'\ntype: {type(agg.index).__name__}')

# Aplanar a DataFrame normal
flat = agg.reset_index()
print('\nflat (DataFrame):')
print(flat)

## 6️⃣ Cuándo aplanar

- **Para CSV de salida**: cliente final espera tabla rectangular.
- **Para plot**: matplotlib/seaborn esperan columnas, no niveles.
- **Para scikit-learn**: features son columnas planas.
- **Para joins**: merge funciona mejor con index plano.

**Cuándo dejar MultiIndex**: análisis interactivo donde el slicing por nivel es frecuente.

## ✅ Checklist

- [ ] Sé construir MultiIndex con tuples/arrays/from_product
- [ ] Indexo con `.loc[('a', 'b')]` y `df.xs(v, level=...)`
- [ ] Convierto wide ↔ long con `unstack`/`stack`
- [ ] Reconozco que groupby con N keys devuelve MultiIndex
- [ ] Aplano con `reset_index` cuando aporta

## 📝 Homework

Ver `README.md`. Ventas trimestre×región, accesos por nivel, unstack/stack, groupby penguins.

## 📖 Definiciones y características

**`MultiIndex`**

Índice jerárquico con N niveles. Cada fila identificada por tupla de N labels (`('España', 2024)`). Útil cuando los datos tienen estructura natural (país→ciudad, año→mes).

**Nivel (level)**

Cada "capa" del MultiIndex. Se referencia por nombre (`level='año'`) o posición (`level=0`). Útil en operaciones como `unstack(level=...)`.

**`stack` / `unstack`**

Mueven niveles entre filas y columnas. **`unstack`** sube un nivel del index a columnas (long→wide). **`stack`** baja un nivel de columnas al index (wide→long). Reversibles.

**`xs` (cross-section)**

Slice por un valor en un nivel: `df.xs(2024, level='año')`. Más limpio que indexar con tuplas parciales.

**Aplanar (flatten)**

Convertir MultiIndex a Index plano: `df.reset_index()` (vuelve a default 0..N) o `df.index = ['_'.join(map(str, t)) for t in df.index]` (concatena niveles).

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `KeyError` al acceder `df.loc['España', 2024]` | loc con MultiIndex requiere **tupla**: `df.loc[('España', 2024)]`. Sin paréntesis pandas lo lee como (fila, columna). |
| `unstack()` lanza `ValueError: Index contains duplicate entries` | Tienes filas duplicadas en (index, columns) → no se puede pivotar. **Fix**: agrega antes (`groupby` con sum/mean) o usa `pivot_table`. |
| `groupby([a, b]).sum()` devuelve cosa extraña con MultiIndex | Es **correcto**: groupby con N keys devuelve MultiIndex. **Si quieres DataFrame plano**: `.reset_index()` después. |
| Plot ignora niveles del MultiIndex | matplotlib/seaborn esperan columnas planas. **Fix**: aplana con `reset_index()` o `unstack()` antes de plotear. |
| `sort_index()` ordena raro con MultiIndex | Default ordena por todos los niveles. Para ordenar por uno específico: `sort_index(level='año')` o `sort_values('col')` si tienes una columna de criterio. |

## ❓ Preguntas frecuentes

**❓ ¿Cuándo MultiIndex aporta vs cuándo complica?**

**Aporta** en análisis interactivo con slicing por nivel frecuente. **Complica** para export a CSV, plot, sklearn — aplana ahí.

**❓ ¿`set_index([a, b])` vs `groupby([a, b])`?**

`set_index` solo mueve cols al index (sin agregar). `groupby` colapsa filas por las cols (con sum/mean/agg). Diferentes operaciones.

**❓ ¿Cómo evito MultiIndex en groupby?**

`groupby([a, b], as_index=False)` devuelve DataFrame plano directamente. O `.reset_index()` después.

**❓ ¿`stack(future_stack=True)` qué significa?**

Es el comportamiento del nuevo stack (default en pandas 3+). Maneja NaN distinto al legacy. Mejor pasarlo siempre explícito para suprimir warnings.

**❓ ¿Performance MultiIndex vs Index plano?**

MultiIndex tiene overhead. Para datasets grandes (>1M filas) con acceso intenso, aplana al final del pipeline.

## 🔗 Referencias

- VanderPlas cap. 3 § 3.6
- [MultiIndex guide](https://pandas.pydata.org/docs/user_guide/advanced.html)

➡️ **Siguiente:** [027 — concat, merge, join](../027-pandas-concat-merge-join/README.md)

## ✅ Soluciones de los ejercicios

A continuación, cada ejercicio de la sección `🧪 Ejercicios` del README resuelto y comentado. Todo el código es **ejecutable sin conexión** (datos sintéticos) e incluye `assert`/`print` para que compruebes el resultado. Intenta resolverlos por tu cuenta antes de mirar la solución.

**Ej. 1 — MultiIndex desde tuplas.**

In [ ]:
import pandas as pd, numpy as np
idx = pd.MultiIndex.from_tuples(
    [('Espana', 2023), ('Espana', 2024), ('Chile', 2023), ('Chile', 2024)],
    names=['pais', 'anio'])
df = pd.DataFrame({'ventas': [100, 120, 80, 95], 'clientes': [10, 12, 8, 9]}, index=idx)
print(df)
assert df.index.nlevels == 2

**Ej. 2 — `from_product`** (mismo índice, menos código).

In [ ]:
idx2 = pd.MultiIndex.from_product([['Espana', 'Chile'], [2023, 2024]], names=['pais', 'anio'])
df2 = pd.DataFrame({'ventas': [100, 120, 80, 95], 'clientes': [10, 12, 8, 9]}, index=idx2)
assert df2.index.equals(df.index)
print('from_product genera el mismo indice que from_tuples.')

**Ej. 3 — Acceso jerárquico** (`loc`, tupla, `xs`).

In [ ]:
print('loc["Espana"]:\n', df.loc['Espana'])
print('loc[("Espana", 2024)]:\n', df.loc[('Espana', 2024)])
print('xs(2024, level="anio"):\n', df.xs(2024, level='anio'))
assert df.loc[('Espana', 2024), 'ventas'] == 120

**Ej. 4 — `unstack` y `stack`.**

In [ ]:
wide = df['ventas'].unstack(level='anio')   # anios como columnas
print(wide)
largo = wide.stack()                          # de vuelta a MultiIndex
assert wide.loc['Espana', 2024] == 120 and largo.shape[0] == 4

**Ej. 5 — `groupby` produce MultiIndex** -> aplanar con `reset_index`.

In [ ]:
import numpy as np, pandas as pd

def make_penguins(seed=42, with_na=False):
    """DataFrame sintetico estilo Palmer Penguins (344 filas), sin internet."""
    rng = np.random.default_rng(seed)
    cfg = {  # especie: (n, islas, bill_len, bill_depth, flipper, body_mass)
        'Adelie':    (152, ['Torgersen', 'Biscoe', 'Dream'], 38.8, 18.3, 190, 3700),
        'Chinstrap': (68,  ['Dream'],                        48.8, 18.4, 196, 3733),
        'Gentoo':    (124, ['Biscoe'],                       47.5, 15.0, 217, 5076),
    }
    filas = []
    for sp, (n, islas, bl, bd, fl, bm) in cfg.items():
        for _ in range(n):
            sex = rng.choice(['male', 'female'])
            k = 1.0 if sex == 'male' else 0.93
            filas.append({
                'species': sp,
                'island': rng.choice(islas),
                'bill_length_mm': round(float(rng.normal(bl, 2.5)), 1),
                'bill_depth_mm': round(float(rng.normal(bd, 1.2)), 1),
                'flipper_length_mm': float(round(rng.normal(fl, 6))),
                'body_mass_g': float(round(rng.normal(bm * k, 300))),
                'sex': sex,
            })
    df = pd.DataFrame(filas)
    if with_na:
        idx = rng.choice(df.index, size=12, replace=False)
        df.loc[idx[:6], 'bill_length_mm'] = np.nan
        df.loc[idx[6:], 'sex'] = np.nan
    return df

peng = make_penguins()
g = peng.groupby(['species', 'sex'])['body_mass_g'].mean()
print('niveles del indice del groupby:', g.index.nlevels)
plano = g.reset_index()
assert plano.shape[1] == 3 and g.index.nlevels == 2
print(plano)